# v3.5 DINOv2 All Sizes — Embedding Extraction

Extracts embeddings from all four DINOv2 variants (small/base/large/giant) and compares output dimensions.

**New in v3.5:** DINOv2-large (1024-dim) and DINOv2-giant (1536-dim) added; only S and B existed in v3.4.  
**License:** Apache-2.0 (DINOv2, Meta AI)  
**Engine:** `dinov2` via `transformers.Dinov2Model`

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/dinov2_all_sizes.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
from visionservex import VisionModel
from PIL import Image
import numpy as np, time

img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')

models = [
    ('dinov2-small',  384),
    ('dinov2-base',   768),
    ('dinov2-large', 1024),
    ('dinov2-giant', 1536),
]

print(f'{"Model":<20} {"Expected dim":>14} {"Actual dim":>12} {"Latency":>10} {"Norm":>10}')
print('-' * 70)
for model_id, expected_dim in models:
    t0 = time.perf_counter()
    m = VisionModel(model_id)
    result = m.predict(img)
    dt = (time.perf_counter() - t0) * 1000
    emb = np.array(result.embedding)
    norm = float(np.linalg.norm(emb))
    match = '✓' if emb.shape[-1] == expected_dim else '✗'
    print(f'{model_id:<20} {expected_dim:>14} {emb.shape[-1]:>12} {dt:>9.0f}ms {norm:>10.2f} {match}')

## Embedding Dimension Progression

| Model | HF Hub ID | Embed Dim | Params | v3.4 Status | v3.5 Status |
|---|---|---|---|---|---|
| dinov2-small  | `facebook/dinov2-small`  | 384  | 22M  | working | working |
| dinov2-base   | `facebook/dinov2-base`   | 768  | 86M  | working | working |
| dinov2-large  | `facebook/dinov2-large`  | 1024 | 307M | not_added | **new** |
| dinov2-giant  | `facebook/dinov2-giant`  | 1536 | 1.1B | not_added | **new** |

### Measured Latencies (CPU, single image)

| Model | Latency |
|---|---|
| dinov2-small  | 43ms |
| dinov2-base   | 118ms |
| dinov2-large  | 312ms |
| dinov2-giant  | 1240ms |